## 1. 背景
我们现在有了一个经过标准化的单细胞测序数据，其保留了细胞差异，同时消除了技术带来的采样误差。通常来说，我们经过上游的mapping（比对），我们会得到30,000到50,000个不等的基因。但是，我们只在第一步的质控中，删除了一小部分基因（少于20个细胞表达的基因）。但实际上，一个细胞表达的基因大约是3,000个左右。意味着我们的测序数据中的一大部分基因可能不具有与单细胞数据相关的生物学意义，这些基因大多数包含0计数，或者是在许多细胞中普遍出现的基因。

故在预处理环节，我们需要计算高可变基因（特征基因），排除那些不具有分析意义的基因，避免影响我们下游的建模以及生物学检测。

In [ ]:
在本章中，我们将介绍三种不同的特征基因选择：

基于基因离散度，基于基因归一化方差以及基于基因的皮尔森残差。

在传统的分析流程中，我们会采用基于基因离散度的方式去计算高可变基因，一般来说，我们首先确定了单细胞数据集中变异最大的一组基因。

我们计算了所有单细胞中每个基因的平均值和离散度（方差/平均值），并根据基因的平均表达将其分为 20 个箱。

然后，在每个箱内，我们对箱内所有基因的离散度进行z归一化，以识别表达值高度可变的基因。



## 2. 基于基因离散度¶
我们首先设置我们的环境。

In [1]:
import omicverse as ov
import scanpy as sc

ov.ov_plot_set()

/home/zhen/miniforge3/envs/scanpy_env/lib/python3.12/site-packages/anndata/utils.py:434: FutureWarning: Importing read_loom from `anndata` is deprecated. Import anndata.io.read_loom instead.
  warnings.warn(msg, FutureWarning)


🔬 Starting plot initialization...
🧬 Detecting GPU devices…
✅ NVIDIA CUDA GPUs detected: 1
    • [CUDA 0] NVIDIA GeForce RTX 3060 Laptop GPU
      Memory: 6.0 GB | Compute: 8.6

   ____            _     _    __                  
  / __ \____ ___  (_)___| |  / /__  _____________ 
 / / / / __ `__ \/ / ___/ | / / _ \/ ___/ ___/ _ \ 
/ /_/ / / / / / / / /__ | |/ /  __/ /  (__  )  __/ 
\____/_/ /_/ /_/_/\___/ |___/\___/_/  /____/\___/                                              

🔖 Version: 1.7.9   📚 Tutorials: https://omicverse.readthedocs.io/
✅ plot_set complete.



In [15]:
adata = sc.read(
    filename="s4d8_quality_control.h5ad",
    backup_url="https://ndownloader.figshare.com/files/40014331",
)

In [16]:
#存储原始数据以便后续还原
ov.utils.store_layers(adata,layers='counts')
adata.layers['counts']=adata.X.copy()

sc.pp.normalize_total(adata)
sc.pp.log1p(adata)
adata

......The X of adata have been stored in counts
normalizing counts per cell
    finished (0:00:00)


AnnData object with n_obs × n_vars = 14814 × 20171
    obs: 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_20_genes', 'total_counts_mt', 'log1p_total_counts_mt', 'pct_counts_mt', 'total_counts_ribo', 'log1p_total_counts_ribo', 'pct_counts_ribo', 'total_counts_hb', 'log1p_total_counts_hb', 'pct_counts_hb', 'outlier', 'mt_outlier', 'scDblFinder_score', 'scDblFinder_class'
    var: 'gene_ids', 'feature_types', 'genome', 'mt', 'ribo', 'hb', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts', 'n_cells'
    uns: 'layers_counts', 'log1p'
    layers: 'counts', 'soupX_counts'

接下来，我们调用scanpy包里的pp.highly_variable_genes函数来计算高可变基因，由于我们使用的是基于基因离散度的方法，故我们需要设置flavor='seurat'，该方法也是默认方法。基于基因离散度的方法寻找高变基因有两个途径：

指定目的高变基因数
指定离散度，平均值过滤高变基因
在这里，我们分别尝试两种不同的方法，首先是基于指定高变基因数的方法：



2.1 指定高可变基因数
由于是教程演示，所以我们不希望改变原来的anndata对象的.var，故我们设置inplace=False，旨在观察和对比高可变基因

In [17]:
adata_dis_num=sc.pp.highly_variable_genes(
    adata,
    flavor="seurat",
    n_top_genes=2000,
    subset=False,
    inplace=False,
)
adata_dis_num

extracting highly variable genes
    finished (0:00:00)


,means,dispersions,mean_bin,dispersions_norm,highly_variable
AL627309.1,0.000955,0.001085,"(-0.00329, 0.186]",-1.267316,False
AL627309.5,0.006057,0.715322,"(-0.00329, 0.186]",1.484390,True
LINC01409,0.036619,0.275313,"(-0.00329, 0.186]",-0.210809,False
LINC01128,0.022142,0.268974,"(-0.00329, 0.186]",-0.235233,False
LINC00115,0.002392,-0.093762,"(-0.00329, 0.186]",-1.632726,False
...,...,...,...,...,...
AC011043.1,0.003747,0.063567,"(-0.00329, 0.186]",-1.026594,False
AL354822.1,0.003144,0.307859,"(-0.00329, 0.186]",-0.085421,False
AL592183.1,0.062956,0.448886,"(-0.00329, 0.186]",0.457906,False
AC240274.1,0.012018,0.158723,"(-0.00329, 0.186]",-0.659990,False


In [18]:
adata_dis_num['highly_variable'].value_counts()

highly_variable
False    18171
True      2000
Name: count, dtype: int64

2.2 指定基因离散度与平均值阈值
除了指定高可变基因数量外，我们还可以通过基因的离散度与平均值的阈值，去计算高可变基因.

<div style="background-color: #5c8c03;">
但是，这种启发式的方法是数据敏感的，阈值所计算出来的高可变基因可能并不是全部高可变。

</div>


<div style="background-color: #ed00f5;">
故基于阈值的方法在现在的分析中应用较少 </div>

In [19]:
adata_dis_cutoff=sc.pp.highly_variable_genes(
    adata,
    flavor="seurat",
    min_disp=0.5,
    min_mean=0.0125,
    max_mean=3,
    subset=False,
    inplace=False,
)
adata_dis_cutoff['highly_variable'].value_counts()

extracting highly variable genes
    finished (0:00:00)


highly_variable
False    17291
True      2880
Name: count, dtype: int64

通过简单的计算我们发现，一共有2,880个高可变基因被选择了出来，一般2000-3000内的高可变基因数都是能接受的 



因为非高可变基因在下游分析时会被过滤掉，我们会将归一化值给保存进raw文件，但这有个缺陷，我们将会永远失去非高可变基因的原始count。因此可以使用释放函数 ov.utils.retrieve_layers，与存放在adata.layers中的数据不同，

<div style="background-color: #560c4e; padding: 10px;">
retrieve_layers函数还可还原全基因的原始计数，我们以下将进行一个简单的对比实验。
</div>

In [20]:
#计算高可变基因（覆盖，inplace=True）
adata.raw = adata
# adata = adata.raw 
sc.pp.highly_variable_genes(
    adata,
    flavor="seurat",
)


extracting highly variable genes
    finished (0:00:01)
--> added
    'highly_variable', boolean vector (adata.var)
    'means', float vector (adata.var)
    'dispersions', float vector (adata.var)
    'dispersions_norm', float vector (adata.var)


In [21]:
#只保留高可变基因
adata = adata[:, adata.var.highly_variable]
print('shape: ',adata.shape)


shape:  (14814, 2880)


In [22]:
adata_counts=adata.copy()
ov.utils.retrieve_layers(adata_counts,layers='counts')
print('normalize adata:',adata.X.max())


......The X of adata have been stored in raw
......The layers counts of adata have been retreved
normalize adata: 4.9448869840604806


In [23]:
print('raw count adata:',adata_counts.X.max())


raw count adata: 694.0


In [24]:
print('shape: ',adata_counts.shape)

shape:  (14814, 2880)


In [ ]:
adata = sc.read(
    filename="s4d8_quality_control.h5ad",
    backup_url="https://ndownloader.figshare.com/files/40014331",
)

In [25]:
#计算高可变基因（覆盖，inplace=True）
 
sc.pp.highly_variable_genes(
    adata,
    flavor="seurat",
)
#只保留高可变基因
adata = adata[:, adata.var.highly_variable]
print('shape: ',adata.shape)

adata_counts=adata.copy()
ov.utils.retrieve_layers(adata_counts,layers='counts')
print('normalize adata:',adata.X.max())
print('raw count adata:',adata_counts.X.max())
print('shape: ',adata_counts.shape)

extracting highly variable genes
    finished (0:00:00)
--> added
    'highly_variable', boolean vector (adata.var)
    'means', float vector (adata.var)
    'dispersions', float vector (adata.var)
    'dispersions_norm', float vector (adata.var)
shape:  (14814, 451)
......The X of adata have been stored in raw
......The layers counts of adata have been retreved
normalize adata: 4.9448869840604806
raw count adata: 694.0
shape:  (14814, 451)
